In [1]:
import sys
import re
import numpy as np
import pandas as pd
import sympy as sp

from sympy.parsing.sympy_parser import (
    parse_expr, standard_transformations,
    implicit_multiplication_application, convert_xor
)
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds

from PySide6.QtCore import Qt
from PySide6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QLayout, QVBoxLayout, QHBoxLayout,
    QGridLayout, QLabel, QLineEdit, QTextEdit, QPushButton, QComboBox,
    QCheckBox, QTableWidget, QTableWidgetItem, QSplitter, QGroupBox,
    QMessageBox, QDoubleSpinBox, QTabWidget, QHeaderView, QFrame
)
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
import matplotlib.pyplot as plt


# ------------------------------------------------------------------
# Defensive layout fix
# ------------------------------------------------------------------
# Qt's addWidget() accepts QWidget objects only. If a QLayout is passed
# by mistake anywhere in the UI construction, put that layout inside a
# QWidget container before forwarding it to the real addWidget().
# This prevents: QBoxLayout.addWidget(QVBoxLayout) TypeError.
_QtVBoxLayout = QVBoxLayout
_QtHBoxLayout = QHBoxLayout


class _SafeAddWidgetMixin:
    def addWidget(self, widget, *args, **kwargs):
        if isinstance(widget, QLayout):
            # If this layout already belongs to a QWidget, add that owning
            # QWidget instead. This is the correct recovery for code such as
            # header.addWidget(title_layout) where title_layout was created
            # with title_widget as its parent.
            owner = widget.parent()
            if isinstance(owner, QWidget):
                widget = owner
            else:
                # Otherwise detach it and install it in a wrapper widget.
                if isinstance(owner, QLayout):
                    owner.removeItem(widget)
                widget.setParent(None)
                container = QWidget()
                container.setLayout(widget)
                widget = container

            print(
                "[Optimization Studio] Recovered an addWidget(QLayout) call; "
                "the layout is now hosted by a QWidget."
            )
        return super().addWidget(widget, *args, **kwargs)


class QVBoxLayout(_SafeAddWidgetMixin, _QtVBoxLayout):
    pass


class QHBoxLayout(_SafeAddWidgetMixin, _QtHBoxLayout):
    pass


TRANSFORMS = standard_transformations + (
    implicit_multiplication_application, convert_xor
)
SAFE_LOCALS = {
    "sqrt": sp.sqrt, "sin": sp.sin, "cos": sp.cos, "tan": sp.tan,
    "exp": sp.exp, "log": sp.log, "ln": sp.log, "abs": sp.Abs,
    "pi": sp.pi, "E": sp.E
}
PALETTE = ["#2563eb", "#dc2626", "#16a34a", "#9333ea",
           "#ea580c", "#0891b2", "#be185d", "#4d7c0f"]


def parse_math(text):
    text = text.strip().replace("−", "-").replace("≤", "<=").replace("≥", ">=")
    if not text:
        raise ValueError("Expression cannot be empty.")
    return parse_expr(text, transformations=TRANSFORMS,
                      local_dict=SAFE_LOCALS, evaluate=True)


def parse_constraint(line):
    for op in ("<=", ">=", "=="):
        if op in line:
            a, b = line.split(op, 1)
            return parse_math(a), parse_math(b), op
    if "=" in line:
        a, b = line.split("=", 1)
        return parse_math(a), parse_math(b), "="
    raise ValueError(f"Invalid constraint: {line}")


def variables_for(obj, cons):
    symbols = set(obj.free_symbols)
    for lhs, rhs, _ in cons:
        symbols |= lhs.free_symbols | rhs.free_symbols
    if not symbols:
        raise ValueError("No decision variables found.")
    return sorted(symbols, key=str)


def is_linear(expr, variables):
    try:
        return sp.Poly(sp.expand(expr), *variables).total_degree() <= 1
    except Exception:
        return False


def linear_coefficients(expr, variables):
    expr = sp.expand(expr)
    poly = sp.Poly(expr, *variables)
    if poly.total_degree() > 1:
        raise ValueError(f"Expected a linear expression: {expr}")
    coeff = np.array([float(expr.coeff(v)) for v in variables])
    const = float(expr.subs({v: 0 for v in variables}))
    return coeff, const


def linear_matrices(obj, cons, variables):
    c, c0 = linear_coefficients(obj, variables)
    aub, bub, aeq, beq = [], [], [], []
    for lhs, rhs, op in cons:
        row, constant = linear_coefficients(lhs-rhs, variables)
        rhsval = -constant
        if op == "<=":
            aub.append(row); bub.append(rhsval)
        elif op == ">=":
            aub.append(-row); bub.append(-rhsval)
        else:
            aeq.append(row); beq.append(rhsval)
    return (c, np.array(aub) if aub else None,
            np.array(bub) if bub else None,
            np.array(aeq) if aeq else None,
            np.array(beq) if beq else None)


def solve_lp(obj, cons, variables, maximize, bounds):
    c, aub, bub, aeq, beq = linear_matrices(obj, cons, variables)
    result = linprog(-c if maximize else c, A_ub=aub, b_ub=bub,
                     A_eq=aeq, b_eq=beq, bounds=bounds, method="highs")
    if not result.success:
        raise RuntimeError(result.message)
    x = result.x
    return x, float(obj.subs(dict(zip(variables, x)))), "HiGHS LP"


def solve_milp(obj, cons, variables, maximize, bounds, mode, integer_names):
    c, aub, bub, aeq, beq = linear_matrices(obj, cons, variables)
    rows, lows, highs = [], [], []
    if aub is not None:
        rows.extend(aub); lows.extend([-np.inf]*len(aub)); highs.extend(bub)
    if aeq is not None:
        rows.extend(aeq); lows.extend(beq); highs.extend(beq)
    lc = LinearConstraint(np.asarray(rows), np.asarray(lows), np.asarray(highs)) if rows else None

    lb = np.array([-np.inf if b[0] is None else b[0] for b in bounds], float)
    ub = np.array([np.inf if b[1] is None else b[1] for b in bounds], float)

    integrality = np.zeros(len(variables), dtype=int)
    names = [str(v) for v in variables]
    if mode in ("Integer Programming", "Binary Programming"):
        integrality[:] = 1
    else:
        for name in integer_names:
            if name not in names:
                raise ValueError(f"Unknown integer variable: {name}")
            integrality[names.index(name)] = 1
        if not np.any(integrality):
            raise ValueError("Enter at least one integer variable for MILP, e.g. x, z.")

    if mode == "Binary Programming":
        lb = np.maximum(lb, 0)
        ub = np.minimum(ub, 1)

    result = milp(c=(-c if maximize else c), integrality=integrality,
                  bounds=Bounds(lb, ub), constraints=lc)
    if not result.success:
        raise RuntimeError(result.message)
    x = result.x
    return x, float(obj.subs(dict(zip(variables, x)))), "HiGHS MILP"


def solve_nlp(obj, cons, variables, maximize, bounds):
    f = sp.lambdify(variables, obj, modules="numpy")
    def fun(x):
        val = float(np.asarray(f(*x)))
        return -val if maximize else val

    scipy_cons = []
    for lhs, rhs, op in cons:
        fn = sp.lambdify(variables, lhs-rhs, modules="numpy")
        if op == "<=":
            scipy_cons.append({"type": "ineq", "fun": lambda x, fn=fn: -float(fn(*x))})
        elif op == ">=":
            scipy_cons.append({"type": "ineq", "fun": lambda x, fn=fn: float(fn(*x))})
        else:
            scipy_cons.append({"type": "eq", "fun": lambda x, fn=fn: float(fn(*x))})

    starts = []
    for k in range(6):
        x0 = []
        for lo, hi in bounds:
            if lo is not None and hi is not None and np.isfinite(lo) and np.isfinite(hi):
                x0.append((lo+hi)/2 if k == 0 else lo + (hi-lo)*(k % 5)/4)
            elif lo is not None and np.isfinite(lo):
                x0.append(lo + [0.5, 1, 2, 5, 10, 20][k])
            elif hi is not None and np.isfinite(hi):
                x0.append(hi - [0.5, 1, 2, 5, 10, 20][k])
            else:
                x0.append([0.5, 1, 2, 5, 10, 20][k])
        starts.append(np.array(x0, float))

    best = None
    for start in starts:
        res = minimize(fun, start, method="SLSQP", bounds=bounds,
                       constraints=scipy_cons,
                       options={"maxiter": 3000, "ftol": 1e-9})
        if res.success and np.all(np.isfinite(res.x)):
            if best is None or res.fun < best.fun:
                best = res
    if best is None:
        raise RuntimeError("No successful local solution. Check feasibility, bounds, or starting points.")
    x = best.x
    return x, float(obj.subs(dict(zip(variables, x)))), "SLSQP (local nonlinear solver)"


def verify_constraints(cons, variables, x):
    sub = dict(zip(variables, x))
    rows = []
    all_ok = True
    for lhs, rhs, op in cons:
        lv, rv = float(lhs.subs(sub)), float(rhs.subs(sub))
        if op == "<=": ok = lv <= rv + 1e-5
        elif op == ">=": ok = lv >= rv - 1e-5
        else: ok = abs(lv-rv) <= 1e-5
        all_ok &= ok
        rows.append((f"{lhs} {op} {rhs}", f"{lv:.6g}", f"{rv:.6g}",
                     "Satisfied" if ok else "Violated"))
    return rows, all_ok


class PlotCanvas(FigureCanvas):
    def __init__(self):
        self.fig = Figure(figsize=(7, 5), dpi=100, tight_layout=True)
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)


class OptimizationStudio(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Optimization Studio | Desktop")
        self.resize(1450, 900)
        self.history = []
        self._build_ui()
        self._apply_style()

    def _build_ui(self):
        root = QWidget()
        self.setCentralWidget(root)
        main = QHBoxLayout(root)
        main.setContentsMargins(0, 0, 0, 0)
        main.setSpacing(0)

        sidebar = QFrame()
        sidebar.setObjectName("sidebar")
        sidebar.setFixedWidth(260)
        side = QVBoxLayout(sidebar)
        brand = QLabel("OPTIMIZATION\nSTUDIO")
        brand.setObjectName("brand")
        side.addWidget(brand)
        side.addWidget(QLabel("MATHEMATICAL WORKSPACE"))
        self.method = QComboBox()
        self.method.addItems([
            "Automatic Model Detection", "Linear Programming",
            "Nonlinear Programming", "Quadratic Programming",
            "Integer Programming", "Binary Programming",
            "Mixed-Integer Programming"
        ])
        side.addWidget(QLabel("OPTIMIZATION METHOD"))
        side.addWidget(self.method)
        self.goal = QComboBox()
        self.goal.addItems(["Maximize", "Minimize"])
        side.addWidget(QLabel("OBJECTIVE"))
        side.addWidget(self.goal)
        self.nonnegative = QCheckBox("All variables non-negative")
        self.nonnegative.setChecked(True)
        side.addWidget(self.nonnegative)
        side.addSpacing(10)
        side.addWidget(QLabel("VARIABLE BOUNDS"))
        self.lower = QLineEdit()
        self.lower.setPlaceholderText("Lower: x=0, y=0")
        self.upper = QLineEdit()
        self.upper.setPlaceholderText("Upper: x=100, y=100")
        side.addWidget(self.lower)
        side.addWidget(self.upper)
        self.integer_vars = QLineEdit()
        self.integer_vars.setPlaceholderText("MILP integers: x, z")
        side.addWidget(self.integer_vars)
        side.addStretch()
        note = QLabel("LP / MILP: HiGHS\nNLP / QP: SLSQP\n2D graph: two variables")
        note.setObjectName("sideNote")
        side.addWidget(note)

        main.addWidget(side)

        content = QWidget()
        layout = QVBoxLayout(content)
        layout.setContentsMargins(26, 22, 26, 22)
        layout.setSpacing(14)

        header = QHBoxLayout()
        title_widget = QWidget()
        title_col = QVBoxLayout(title_widget)
        title_col.setContentsMargins(0, 0, 0, 0)
        title = QLabel("Optimization workspace")
        title.setObjectName("pageTitle")
        subtitle = QLabel("Build, solve and inspect mathematical models.")
        subtitle.setObjectName("subtitle")
        title_col.addWidget(title)
        title_col.addWidget(subtitle)
        # Add the QWidget container to the header (not the QVBoxLayout itself).
        header.addWidget(title_widget)
        header.addStretch()
        self.status = QLabel("READY")
        self.status.setObjectName("status")
        header.addWidget(self.status)
        layout.addLayout(header)

        input_row = QHBoxLayout()
        obj_box = QGroupBox("OBJECTIVE FUNCTION")
        ob = QVBoxLayout(obj_box)
        self.model_name = QLineEdit("Model 1")
        self.model_name.setPlaceholderText("Model name")
        self.objective = QLineEdit("40x + 30y")
        self.objective.setPlaceholderText("Example: 7x + 5y")
        ob.addWidget(QLabel("Model name")); ob.addWidget(self.model_name)
        ob.addWidget(QLabel("Write naturally — 7x, x^2, sqrt(x)")); ob.addWidget(self.objective)

        cons_box = QGroupBox("CONSTRAINTS")
        cb = QVBoxLayout(cons_box)
        self.constraints = QTextEdit("2x + y <= 100\nx + 2y <= 80")
        self.constraints.setPlaceholderText("One constraint per line\n2x + 3y <= 100\nx + y = 50")
        cb.addWidget(self.constraints)
        input_row.addWidget(obj_box, 1); input_row.addWidget(cons_box, 1)
        layout.addLayout(input_row)

        buttons = QHBoxLayout()
        self.solve_btn = QPushButton("▶  SOLVE MODEL")
        self.solve_btn.setObjectName("solveButton")
        self.solve_btn.clicked.connect(self.solve)
        self.clear_btn = QPushButton("Clear comparison history")
        self.clear_btn.clicked.connect(self.clear_history)
        self.export_btn = QPushButton("Export results CSV")
        self.export_btn.clicked.connect(self.export_csv)
        buttons.addWidget(self.solve_btn); buttons.addWidget(self.clear_btn); buttons.addWidget(self.export_btn)
        buttons.addStretch()
        layout.addLayout(buttons)

        self.tabs = QTabWidget()
        results_tab = QWidget()
        rt = QVBoxLayout(results_tab)
        self.metrics = QLabel("Run a model to see the optimal solution.")
        self.metrics.setObjectName("metrics")
        rt.addWidget(self.metrics)
        self.solution_table = QTableWidget(0, 2)
        self.solution_table.setHorizontalHeaderLabels(["Decision variable", "Optimal value"])
        self.solution_table.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        rt.addWidget(QLabel("DECISION VARIABLES"))
        rt.addWidget(self.solution_table)
        self.constraint_table = QTableWidget(0, 4)
        self.constraint_table.setHorizontalHeaderLabels(["Constraint", "LHS", "RHS", "Status"])
        self.constraint_table.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        rt.addWidget(QLabel("CONSTRAINT VERIFICATION"))
        rt.addWidget(self.constraint_table)
        self.tabs.addTab(results_tab, "Results")

        graph_tab = QWidget()
        gt = QVBoxLayout(graph_tab)
        range_row = QHBoxLayout()
        self.xmin = QDoubleSpinBox(); self.xmin.setRange(-1e6, 1e6); self.xmin.setValue(0)
        self.xmax = QDoubleSpinBox(); self.xmax.setRange(-1e6, 1e6); self.xmax.setValue(100)
        self.ymin = QDoubleSpinBox(); self.ymin.setRange(-1e6, 1e6); self.ymin.setValue(0)
        self.ymax = QDoubleSpinBox(); self.ymax.setRange(-1e6, 1e6); self.ymax.setValue(100)
        for label, widget in [("X min", self.xmin), ("X max", self.xmax), ("Y min", self.ymin), ("Y max", self.ymax)]:
            range_row.addWidget(QLabel(label)); range_row.addWidget(widget)
        self.refresh_plot_btn = QPushButton("Refresh graph")
        self.refresh_plot_btn.clicked.connect(self.refresh_plot)
        range_row.addWidget(self.refresh_plot_btn)
        gt.addLayout(range_row)
        self.canvas = PlotCanvas()
        gt.addWidget(self.canvas)
        self.tabs.addTab(graph_tab, "Feasible region")

        history_tab = QWidget()
        ht = QVBoxLayout(history_tab)
        self.history_table = QTableWidget(0, 4)
        self.history_table.setHorizontalHeaderLabels(["Model", "Method", "Objective value", "Decision variables"])
        self.history_table.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        ht.addWidget(self.history_table)
        self.tabs.addTab(history_tab, "Model comparison")
        layout.addWidget(self.tabs, 1)
        main.addWidget(content, 1)

    def _apply_style(self):
        self.setStyleSheet("""
            QMainWindow, QWidget { background: #f4f6fa; color: #172033; font-family: Segoe UI; font-size: 10pt; }
            #sidebar { background: #101a2c; color: #e8eef8; }
            #sidebar QLabel { color: #c4d0e0; }
            #brand { color: white; font-size: 22pt; font-weight: 800; padding: 14px 0 22px 0; }
            #sideNote { color: #91a4bd; padding: 12px; background: #1b2940; border-radius: 8px; }
            #pageTitle { font-size: 22pt; font-weight: 700; color: #14213a; }
            #subtitle { color: #64748b; }
            #status { background: #dff7e8; color: #13733b; padding: 7px 14px; border-radius: 10px; font-weight: 700; }
            QGroupBox { background: white; border: 1px solid #dce3ed; border-radius: 10px; margin-top: 12px; padding: 12px; font-weight: 700; color: #475569; }
            QGroupBox::title { subcontrol-origin: margin; left: 12px; padding: 0 5px; }
            QLineEdit, QTextEdit, QComboBox, QDoubleSpinBox { background: white; border: 1px solid #cbd5e1; border-radius: 6px; padding: 8px; color: #172033; }
            QPushButton { background: #e7edf6; border: 0; border-radius: 7px; padding: 10px 16px; font-weight: 600; }
            QPushButton:hover { background: #d6e1ef; }
            #solveButton { background: #1769e0; color: white; padding: 12px 22px; }
            #solveButton:hover { background: #0f55bd; }
            QTabWidget::pane { background: white; border: 1px solid #dce3ed; border-radius: 8px; }
            QTabBar::tab { padding: 10px 18px; background: #e9eef5; border-top-left-radius: 6px; border-top-right-radius: 6px; }
            QTabBar::tab:selected { background: white; color: #1769e0; font-weight: 700; }
            QTableWidget { background: white; border: 1px solid #e2e8f0; gridline-color: #e2e8f0; }
            #metrics { background: #eaf2ff; border: 1px solid #c9ddff; color: #174a91; border-radius: 8px; padding: 14px; font-size: 12pt; font-weight: 700; }
        """)

    def get_bounds(self, variables):
        bounds = [(0, None) if self.nonnegative.isChecked() else (None, None)
                  for _ in variables]
        names = [str(v) for v in variables]
        for text, which in [(self.lower.text(), 0), (self.upper.text(), 1)]:
            if not text.strip():
                continue
            for part in text.split(","):
                if "=" not in part:
                    raise ValueError("Bounds format: x=0, y=10")
                name, val = part.split("=", 1)
                name = name.strip()
                if name not in names:
                    raise ValueError(f"Unknown variable in bounds: {name}")
                i = names.index(name)
                old = list(bounds[i])
                old[which] = float(val)
                bounds[i] = tuple(old)
        for lo, hi in bounds:
            if lo is not None and hi is not None and lo > hi:
                raise ValueError("Lower bound cannot exceed upper bound.")
        return bounds

    def solve(self):
        try:
            obj = parse_math(self.objective.text())
            lines = [s.strip() for s in self.constraints.toPlainText().splitlines() if s.strip()]
            cons = [parse_constraint(s) for s in lines]
            variables = variables_for(obj, cons)
            all_linear = is_linear(obj, variables) and all(is_linear(lhs-rhs, variables) for lhs, rhs, _ in cons)
            try:
                degree = sp.Poly(sp.expand(obj), *variables).total_degree()
            except Exception:
                degree = None

            method = self.method.currentText()
            if method == "Automatic Model Detection":
                if all_linear:
                    method = "Linear Programming"
                elif degree is not None and degree <= 2 and all(is_linear(lhs-rhs, variables) for lhs, rhs, _ in cons):
                    method = "Quadratic Programming"
                else:
                    method = "Nonlinear Programming"

            if method in ("Linear Programming", "Integer Programming",
                          "Binary Programming", "Mixed-Integer Programming") and not all_linear:
                raise ValueError("This optimization method requires linear objective and constraints.")
            if method == "Quadratic Programming":
                if degree is None or degree > 2:
                    raise ValueError("Quadratic Programming requires an objective of degree 2 or less.")
                if not all(is_linear(lhs-rhs, variables) for lhs, rhs, _ in cons):
                    raise ValueError("This QP implementation requires linear constraints.")

            bounds = self.get_bounds(variables)
            maximize = self.goal.currentText() == "Maximize"

            if method == "Linear Programming":
                x, z, solver = solve_lp(obj, cons, variables, maximize, bounds)
            elif method in ("Integer Programming", "Binary Programming", "Mixed-Integer Programming"):
                integer_names = [s.strip() for s in self.integer_vars.text().split(",") if s.strip()]
                x, z, solver = solve_milp(obj, cons, variables, maximize, bounds, method, integer_names)
            else:
                x, z, solver = solve_nlp(obj, cons, variables, maximize, bounds)

            rows, feasible = verify_constraints(cons, variables, x)
            if not feasible:
                raise RuntimeError("Returned solution failed constraint verification.")

            record = {
                "name": self.model_name.text().strip() or f"Model {len(self.history)+1}",
                "method": method, "obj": obj, "cons": cons,
                "variables": variables, "x": np.asarray(x), "z": z
            }
            self.history.append(record)
            self.populate_results(record, solver, rows)
            self.populate_history()
            self.refresh_plot()
            self.status.setText("SOLUTION FOUND")
            self.tabs.setCurrentIndex(0)
        except Exception as e:
            self.status.setText("MODEL ERROR")
            QMessageBox.critical(self, "Optimization error", str(e))

    def populate_results(self, record, solver, rows):
        self.metrics.setText(
            f"Optimal objective: {record['z']:,.8g}     |     {solver}     |     {record['method']}"
        )
        self.solution_table.setRowCount(len(record["variables"]))
        for i, (v, val) in enumerate(zip(record["variables"], record["x"])):
            self.solution_table.setItem(i, 0, QTableWidgetItem(str(v)))
            self.solution_table.setItem(i, 1, QTableWidgetItem(f"{val:.10g}"))
        self.constraint_table.setRowCount(len(rows))
        for i, row in enumerate(rows):
            for j, val in enumerate(row):
                item = QTableWidgetItem(str(val))
                if j == 3:
                    item.setForeground(Qt.darkGreen if val == "Satisfied" else Qt.red)
                self.constraint_table.setItem(i, j, item)

    def populate_history(self):
        self.history_table.setRowCount(len(self.history))
        for i, rec in enumerate(self.history):
            vals = ", ".join(f"{v}={x:.5g}" for v, x in zip(rec["variables"], rec["x"]))
            entries = [rec["name"], rec["method"], f"{rec['z']:.8g}", vals]
            for j, value in enumerate(entries):
                self.history_table.setItem(i, j, QTableWidgetItem(value))

    def refresh_plot(self):
        ax = self.canvas.ax
        ax.clear()
        if not self.history:
            ax.set_title("Feasible regions will appear here")
            ax.grid(alpha=0.25)
            self.canvas.draw()
            return

        xmin, xmax = self.xmin.value(), self.xmax.value()
        ymin, ymax = self.ymin.value(), self.ymax.value()
        if xmin >= xmax or ymin >= ymax:
            QMessageBox.warning(self, "Graph range", "Minimum must be smaller than maximum.")
            return
        xx = np.linspace(xmin, xmax, 350)
        yy = np.linspace(ymin, ymax, 350)
        X, Y = np.meshgrid(xx, yy)

        for idx, rec in enumerate(self.history):
            if len(rec["variables"]) != 2:
                continue
            color = PALETTE[idx % len(PALETTE)]
            v1, v2 = rec["variables"]
            mask = np.ones_like(X, dtype=bool)
            for lhs, rhs, op in rec["cons"]:
                fn = sp.lambdify((v1, v2), lhs-rhs, modules="numpy")
                d = np.asarray(fn(X, Y), dtype=float)
                if np.isscalar(d):
                    d = np.full_like(X, float(d))
                if op == "<=": mask &= d <= 1e-9
                elif op == ">=": mask &= d >= -1e-9
                else: mask &= np.abs(d) <= 0.025
            if np.any(mask):
                ax.contourf(X, Y, mask.astype(float), levels=[0.5, 1.5],
                            colors=[color], alpha=0.17)
            ax.scatter(rec["x"][0], rec["x"][1], marker="*", s=180,
                       color=color, edgecolor="black", linewidth=0.5,
                       label=f"{rec['name']} optimum")
        ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
        ax.set_xlabel(str(self.history[-1]["variables"][0]))
        ax.set_ylabel(str(self.history[-1]["variables"][1]))
        ax.set_title("Feasible regions and optimal solutions (2D models)")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8, loc="best")
        self.canvas.draw()

    def clear_history(self):
        self.history.clear()
        self.populate_history()
        self.refresh_plot()
        self.status.setText("HISTORY CLEARED")

    def export_csv(self):
        if not self.history:
            QMessageBox.information(self, "Export", "Solve at least one model first.")
            return
        from PySide6.QtWidgets import QFileDialog
        path, _ = QFileDialog.getSaveFileName(self, "Export results", "optimization_results.csv", "CSV files (*.csv)")
        if not path:
            return
        rows = []
        for rec in self.history:
            row = {"Model": rec["name"], "Method": rec["method"], "Objective": rec["z"]}
            row.update({str(v): float(x) for v, x in zip(rec["variables"], rec["x"])})
            rows.append(row)
        pd.DataFrame(rows).to_csv(path, index=False)
        QMessageBox.information(self, "Export complete", f"Saved results to:\n{path}")


# Keep references alive when the file is re-run from an interactive IDE/console.
_APP = None
_WINDOW = None


def main():
    global _APP, _WINDOW

    # QApplication is a process-wide singleton. Reuse it if an IDE, notebook,
    # or previous run has already created one.
    app = QApplication.instance()
    created_app = app is None
    if created_app:
        app = QApplication(sys.argv)

    _APP = app
    app.setStyle("Fusion")

    # Close the previous window from a re-run, if it still exists.
    if _WINDOW is not None:
        try:
            _WINDOW.close()
            _WINDOW.deleteLater()
        except RuntimeError:
            pass

    _WINDOW = OptimizationStudio()
    _WINDOW.show()

    # In a normal Windows launch, start the Qt event loop. If an IDE/notebook
    # already owns the QApplication, leave its existing event loop alone.
    if created_app:
        return app.exec()
    return _WINDOW


if __name__ == "__main__":
    main()


[Optimization Studio] Recovered an addWidget(QLayout) call; the layout is now hosted by a QWidget.
